# Upload Existing Conference Run to GitHub
Use this only if the primary run succeeded in Drive but GitHub export was skipped. It does not retrain any model.


In [ ]:
import os, shutil, tempfile, subprocess
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive',force_remount=False)
root=Path('/content/drive/MyDrive/MAT-Appendix/runs')
candidates=sorted([p for p in root.iterdir() if (p/'mat_appendix_complete_reproducibility.pkl').exists()])
if not candidates:
    raise FileNotFoundError('No conference-complete Drive run found.')
source=candidates[-1]; run_id=source.name
token=(userdata.get('GITHUB_TOKEN') or '').strip()
if not token:
    raise RuntimeError('Add GITHUB_TOKEN to Colab Secrets with Contents: read/write.')
work=Path(tempfile.mkdtemp()); repo=work/'repo'; env=os.environ.copy(); env['GITHUB_TOKEN']=token
ask=work/'askpass.sh'; ask.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *Password*) echo "$GITHUB_TOKEN";; esac\n'); ask.chmod(0o700)
env['GIT_ASKPASS']=str(ask); env['GIT_TERMINAL_PROMPT']='0'
subprocess.check_call(['git','clone','--depth','1','https://github.com/AzizulHakim00/MAT-Appendix.git',str(repo)],env=env)
dest=repo/'results'/'runs'/run_id; dest.mkdir(parents=True,exist_ok=True)
for folder in ['tables','figures']:
    shutil.copytree(source/folder,dest/folder,dirs_exist_ok=True)
for name in ['mat_appendix_complete_reproducibility.pkl','paper_results.json','software_versions.json','MODEL_CARD.md','LIMITATIONS.md','manifest.json']:
    shutil.copy2(source/name,dest/name)
(repo/'results'/'LATEST_RUN.txt').write_text(run_id+'\n')
subprocess.check_call(['git','-C',str(repo),'config','user.name','MAT-Appendix Colab'])
subprocess.check_call(['git','-C',str(repo),'config','user.email','actions@users.noreply.github.com'])
subprocess.check_call(['git','-C',str(repo),'add','results'])
subprocess.check_call(['git','-C',str(repo),'commit','-m',f'Upload conference-complete run {run_id}'])
subprocess.check_call(['git','-C',str(repo),'push','origin','main'],env=env)
print('Uploaded run:',run_id)
